# Bauxite risk

### Step 0: import required packages and set up base path

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import fiona
from pathlib import Path
import matplotlib.pyplot as plt
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import os
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.windows import Window
from rasterio.plot import show
import scipy.ndimage as nd
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from glob import glob
import cartopy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
# from analysis_utils import *
import matplotlib
print(matplotlib.rcParams['font.family'])
matplotlib.rcParams['font.family'] = 'Times New Roman'

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_path = base_path / "dphil_paper_3"


In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)
jamaica_boundary['area_hectares'] = 0.0001*jamaica_boundary.geometry.area # Convert area to hectares
jamaica_total_area = jamaica_boundary['area_hectares'].sum()

In [ ]:
landcover = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_landcover.gpkg"
landcover = gpd.read_file(landcover).to_crs(jamaica_metric_grid_crs)
# bauxite = base_path / "dphil_common_cross_cutting/common_incoming_data/bauxite/Bauxite areas.shp"
# bauxite = gpd.read_file(bauxite).to_crs(jamaica_metric_grid_crs)


In [ ]:
nsdmb_path = base_path / "dphil_common_cross_cutting/common_incoming_data/nsdmb/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb"

# List all layers in the geodatabase
layers = fiona.listlayers(nsdmb_path)
#("Available layers in the geodatabase:")
#for layer in layers:
    #print(layer)

# Choose a layer to load (replace 'your_layer_name' with the actual layer name)
layer_name = "bauxite_reserves"

# Read the layer into a GeoDataFrame
bauxite_reserves = gpd.read_file(nsdmb_path, layer=layer_name)

# Inspect the first few rows
display(bauxite_reserves.head())

bauxite_reseves = bauxite_reserves.to_crs(jamaica_metric_grid_crs)
print(bauxite_reserves.crs)

In [ ]:
landcover["area_hectares"] = landcover.geometry.area * 0.0001
landcover_total_area = landcover["area_hectares"].sum()

display(landcover.head())
print("Total area (ha):", landcover_total_area)


### Intersect land use and bauxite and calculate areas

In [ ]:
# Ensure valid geometries and same CRS
landcover = landcover.copy()
landcover = landcover[landcover.geometry.notna()].copy()
landcover["geometry"] = landcover.geometry.make_valid()
landcover = landcover.dropna(subset=["Classify"])

bauxite_reserves = bauxite_reserves.copy()
bauxite_reserves = bauxite_reserves[bauxite_reserves.geometry.notna()].copy()
bauxite_reserves["geometry"] = bauxite_reserves.geometry.make_valid()

# Dissolve bauxite to one non-overlapping footprint
bauxite_reserves_mask = gpd.GeoDataFrame(
    geometry=[bauxite_reserves.geometry.union_all()],
    crs=jamaica_metric_grid_crs
)

# Intersect landcover with bauxite
landcover_bauxite = gpd.clip(
    landcover[["Classify", "geometry"]].copy(),
    bauxite_reserves_mask
)

# Area by land-use class on bauxite
landcover_bauxite["area_m2"] = landcover_bauxite.geometry.area
area_on_bauxite = (
    landcover_bauxite
    .groupby("Classify", as_index=False)["area_m2"]
    .sum()
    .sort_values("area_m2", ascending=False)
)

area_on_bauxite["area_ha"] = area_on_bauxite["area_m2"] / 1e4
area_on_bauxite["area_km2"] = area_on_bauxite["area_m2"] / 1e6

class_total = (
    landcover.assign(total_m2=landcover.geometry.area)
    .groupby("Classify", as_index=False)["total_m2"].sum()
)

area_on_bauxite = area_on_bauxite.merge(class_total, on="Classify", how="left")
area_on_bauxite["% of class on bauxite"] = (
    area_on_bauxite["area_m2"] / area_on_bauxite["total_m2"] * 100
)

display(area_on_bauxite.sort_values("% of class on bauxite", ascending=False))

# display(area_on_bauxite)


### Protected area on bauxite

In [ ]:
# Define class groups (edit names to exactly match your Classify values)
plantation_classes = [
    "Hardwood Plantation: Mahogany",
    "Hardwood Plantation: Mahoe",
    "Hardwood Plantation: Mixed",
    "Plantation: Tree crops, shrub crops, sugar cane"
]

forest_classes = [
    "Closed broadleaved forest (Primary Forest)",
    "Disturbed broadleaved forest (Secondary Forest)",
    "Secondary Forest",
    "Open dry forest - Tall (Woodland/Savanna)",
    "Open dry forest - Short"
]

mixed_forest_and_agriculture_classes = [
    "Fields and Secondary Forest",
    "Fields or Secondary Forest/Pine Plantation",
]

mixed_agriculture_and_bamboo_classes = [ 
    "Fields and Bamboo",
    "Bamboo and Fields"
]

mixed_bamboo_and_forest_classes = [
    "Bamboo and Secondary Forest"
]
]

In [ ]:
# Path to protected areas shapefile
forest_reserves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/Forest_reserves.shp"
protected_areas_path = base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/Protected_areas.shp"

# Load + reproject
forest_reserves = gpd.read_file(forest_reserves_path).to_crs(jamaica_metric_grid_crs)
protected_areas = gpd.read_file(protected_areas_path).to_crs(jamaica_metric_grid_crs)

# Combine geometries only
protected_all = gpd.GeoDataFrame(
    pd.concat(
        [forest_reserves[['geometry']], protected_areas[['geometry']]],
        ignore_index=True
    ),
    crs=jamaica_metric_grid_crs
)

# Clean + dissolve to one non-overlapping geometry
protected_all = protected_all[protected_all.geometry.notna()].copy()
protected_all['geometry'] = protected_all.geometry.make_valid()

combined_protected_layers = gpd.GeoDataFrame(
    geometry=[protected_all.geometry.union_all()],
    crs=jamaica_metric_grid_crs
)

In [ ]:
# CELL: Intersect landcover-on-bauxite with protected mask
landcover_bauxite_protected = gpd.clip(
    landcover_bauxite[["Classify", "geometry"]].copy(),
    combined_protected_layers
)

landcover_bauxite_protected["area_m2_protected"] = landcover_bauxite_protected.geometry.area

area_on_bauxite_protected = (
    landcover_bauxite_protected
    .groupby("Classify", as_index=False)["area_m2_protected"]
    .sum()
)

area_on_bauxite_protected["area_km2_protected"] = area_on_bauxite_protected["area_m2_protected"] / 1e6
display(area_on_bauxite_protected.sort_values("area_m2_protected", ascending=False))


In [ ]:
# CELL: Merge with your existing area_on_bauxite table and compute percentages
area_on_bauxite_with_protected = (
    area_on_bauxite
    .merge(area_on_bauxite_protected, on="Classify", how="left")
    .fillna({"area_m2_protected": 0, "area_km2_protected": 0})
)

# % of each class-on-bauxite that is protected
area_on_bauxite_with_protected["% of bauxite portion protected"] = (
    area_on_bauxite_with_protected["area_m2_protected"]
    / area_on_bauxite_with_protected["area_m2"]
    * 100
)

# Optional: % of whole class that is both bauxite + protected
area_on_bauxite_with_protected["% of class that is bauxite+protected"] = (
    area_on_bauxite_with_protected["area_m2_protected"]
    / area_on_bauxite_with_protected["total_m2"]
    * 100
)

area_on_bauxite_with_protected = area_on_bauxite_with_protected[[
    "Classify",
    "area_km2",                               # area on bauxite
    "% of class on bauxite",
    "area_km2_protected",                     # area on bauxite + protected
    "% of bauxite portion protected",
    "% of class that is bauxite+protected"
]].rename(columns={
    "area_km2": "Area on Bauxite (km²)",
    "area_km2_protected": "Area on Bauxite + Protected (km²)"
})

display(
    area_on_bauxite_with_protected.sort_values(
        "% of bauxite portion protected", ascending=False
    )
)


In [ ]:
# 1) Total forest area in Jamaica
forest_total_m2 = landcover.loc[
    landcover["Classify"].isin(forest_classes),
    "geometry"
].area.sum()

# 2) Forest area on bauxite
forest_on_bauxite_m2 = landcover_bauxite.loc[
    landcover_bauxite["Classify"].isin(forest_classes),
    "geometry"
].area.sum()

# 3) Forest area on bauxite that is also protected
forest_on_bauxite_protected_m2 = landcover_bauxite_protected.loc[
    landcover_bauxite_protected["Classify"].isin(forest_classes),
    "geometry"
].area.sum()

# Percentages
pct_forest_on_bauxite = (forest_on_bauxite_m2 / forest_total_m2 * 100) if forest_total_m2 > 0 else 0
pct_of_forest_on_bauxite_protected = (
    forest_on_bauxite_protected_m2 / forest_on_bauxite_m2 * 100
) if forest_on_bauxite_m2 > 0 else 0

print(f"Forest total area (km²): {forest_total_m2 / 1e6:.2f}")
print(f"Forest on bauxite (km²): {forest_on_bauxite_m2 / 1e6:.2f}")
print(f"Forest on bauxite + protected (km²): {forest_on_bauxite_protected_m2 / 1e6:.2f}")
print(f"% of forest on bauxite: {pct_forest_on_bauxite:.2f}%")
print(f"% of forest-on-bauxite that is protected: {pct_of_forest_on_bauxite_protected:.2f}%")

#### Step 3.2: plot the bauxite reserves to inspect

In [ ]:
# Create a base plot for the Jamaica boundary
fig, ax = plt.subplots(figsize=(10, 10))
jamaica_boundary.plot(ax=ax, color="none", edgecolor="black", linewidth=1, label="Jamaica Boundary")

# Overlay the bauxite reserves
bauxite_reserves.plot(ax=ax, color="grey", edgecolor="black", alpha=0.7, label="Bauxite Reserves")

# Add a title and legend
plt.title("Bauxite Reserves Over Jamaica Boundary")
plt.legend()
plt.show()

#### Step 3.3: intersect bauxite reserves with land use

In [ ]:
# Perform intersection of bauxite reserves and land use
landcover_bauxite = gpd.overlay(bauxite_reserves, landcover, how="intersection")

In [ ]:
# landcover_bauxite.plot()
display(landcover_bauxite.head())

#### Step 3.4: Calculate the how much of Jamaica's area is bauxite reserves

In [ ]:
# Calculate the the total area of bauxite reserves (m2) and (km2) in Jamaica
landcover_bauxite['area_m2'] = landcover_bauxite.geometry.area
landcover_bauxite['area_km2'] = landcover_bauxite['area_m2'] / 1e6

# Ensure the Jamaica boundary is in the correct CRS for area calculations
jamaica_boundary = jamaica_boundary.to_crs(jamaica_metric_grid_crs)

# Calculate the area of Jamaica using the Jamaica boundary (m2)
jamaica_boundary['area_m2'] = jamaica_boundary.geometry.area

# Sum the areas of Jamaica boundary to get the total area
total_area_jamaica_m2 = jamaica_boundary['area_m2'].sum()

# Convert the total area of Jamaica to square kilometers
total_area_jamaica_km2 = total_area_jamaica_m2 / 1e6

# Calculate the percentage of Jamaica's area that is bauxite reserve (first by calculating area of Jamaica boundary)
total_bauxite_area_km2 = landcover_bauxite['area_km2'].sum()  # Sum of all bauxite reserves
percentage_bauxite_in_Jamaica = (total_bauxite_area_km2 / total_area_jamaica_km2) * 100

# Rename columns for clarity
#bauxite_area_summary.columns = ['Land Use Type', 'Area (m²)', 'Percentage (%)', 'Area (km²)']

# Display the results
print(f"Total area of Jamaica: {total_area_jamaica_km2:.2f} km²")
print(f"Total area of bauxite reserves: {total_bauxite_area_km2:.2f} km²")
print(f"Percentage of Jamaica's area that is bauxite reserve: {percentage_bauxite_in_Jamaica:.2f}%")

#### Step 3.5: Calculate how much of each land use's area and what percentage of each land use is on bauxite

In [ ]:
# Calculate the total area of each land use category in Jamaica
terrestrial_landcover['area_m2'] = terrestrial_landcover.geometry.area
land_use_total_area = (
    terrestrial_landcover.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)
land_use_total_area['area_km2'] = land_use_total_area['area_m2'] / 1e6
land_use_total_area.rename(
    columns={'area_m2': 'Total Area in Jamaica (m²)', 'area_km2': 'Total Area in Jamaica (km²)'},
    inplace=True
)

# Calculate the total area of each land use category on bauxite reserves
landcover_bauxite['area_m2'] = landcover_bauxite.geometry.area
bauxite_land_use_area = (
    landcover_bauxite.groupby('Classify')['area_m2']
    .sum()
    .reset_index()
)
bauxite_land_use_area['area_km2'] = bauxite_land_use_area['area_m2'] / 1e6
bauxite_land_use_area.rename(
    columns={'area_m2': 'Area on Bauxite (m²)', 'area_km2': 'Area on Bauxite (km²)'},
    inplace=True
)

# Merge the two summaries (total land use and bauxite land use)
land_use_bauxite_comparison = land_use_total_area.merge(
    bauxite_land_use_area,
    on='Classify',
    how='left'
)

# Fill NaN values for land use categories not found on bauxite reserves
land_use_bauxite_comparison['Area on Bauxite (m²)'] = land_use_bauxite_comparison['Area on Bauxite (m²)'].fillna(0)
land_use_bauxite_comparison['Area on Bauxite (km²)'] = land_use_bauxite_comparison['Area on Bauxite (km²)'].fillna(0)

# Calculate the percentage of Jamaica's total area for each land use category
total_area_jamaica_m2 = jamaica_boundary['area_m2'].sum()
land_use_bauxite_comparison['Percentage of Jamaica\'s Area (%)'] = (
    land_use_bauxite_comparison['Total Area in Jamaica (m²)'] / total_area_jamaica_m2 * 100
)

# Calculate the percentage of each land use category in Jamaica that is found on bauxite reserves
land_use_bauxite_comparison['Percentage on Bauxite (%)'] = (
    land_use_bauxite_comparison['Area on Bauxite (m²)'] /
    land_use_bauxite_comparison['Total Area in Jamaica (m²)'] * 100
)

# Reorder columns for clarity
land_use_bauxite_comparison = land_use_bauxite_comparison[[
    'Classify',
    'Total Area in Jamaica (km²)',
    'Percentage of Jamaica\'s Area (%)',
    'Area on Bauxite (km²)',
    'Percentage on Bauxite (%)'
]]

# Display the summary table
display(land_use_bauxite_comparison)

# Optional: Save the result to a CSV file
# land_use_bauxite_comparison.to_csv("land_use_bauxite_comparison.csv", index=False)

In [ ]:
# Apply the mapping to classify land use into your predefined categories
terrestrial_landcover['Category'] = terrestrial_landcover['Classify'].replace(landuse_category_mapping)

# Calculate the total area of each land use category in Jamaica
terrestrial_landcover['area_m2'] = terrestrial_landcover.geometry.area
land_use_total_area = (
    terrestrial_landcover.groupby('Category')['area_m2']
    .sum()
    .reset_index()
)
land_use_total_area['area_km2'] = land_use_total_area['area_m2'] / 1e6
land_use_total_area.rename(
    columns={'area_m2': 'Total Area in Jamaica (m²)', 'area_km2': 'Total Area in Jamaica (km²)'},
    inplace=True
)

# Calculate the total area of each land use category on bauxite reserves
landcover_bauxite['Category'] = landcover_bauxite['Classify'].replace(landuse_category_mapping)
landcover_bauxite['area_m2'] = landcover_bauxite.geometry.area
bauxite_land_use_area = (
    landcover_bauxite.groupby('Category')['area_m2']
    .sum()
    .reset_index()
)
bauxite_land_use_area['area_km2'] = bauxite_land_use_area['area_m2'] / 1e6
bauxite_land_use_area.rename(
    columns={'area_m2': 'Area on Bauxite (m²)', 'area_km2': 'Area on Bauxite (km²)'},
    inplace=True
)

# Merge the two summaries (total land use and bauxite land use)
land_use_bauxite_comparison = land_use_total_area.merge(
    bauxite_land_use_area,
    on='Category',
    how='left'
)

# Fill NaN values for land use categories not found on bauxite reserves
land_use_bauxite_comparison['Area on Bauxite (m²)'] = land_use_bauxite_comparison['Area on Bauxite (m²)'].fillna(0)
land_use_bauxite_comparison['Area on Bauxite (km²)'] = land_use_bauxite_comparison['Area on Bauxite (km²)'].fillna(0)

# Calculate the percentage of Jamaica's total area for each land use category
total_area_jamaica_m2 = jamaica_boundary['area_m2'].sum()
land_use_bauxite_comparison["Percentage of Jamaica's Area (%)"] = (
    land_use_bauxite_comparison['Total Area in Jamaica (m²)'] / total_area_jamaica_m2 * 100
)

# Calculate the percentage of each land use category in Jamaica that is found on bauxite reserves
land_use_bauxite_comparison['Percentage of land use category on Bauxite (%)'] = (
    land_use_bauxite_comparison['Area on Bauxite (m²)'] /
    land_use_bauxite_comparison['Total Area in Jamaica (m²)'] * 100
)

# Reorder columns for clarity
land_use_bauxite_comparison = land_use_bauxite_comparison[[
    'Category',
    'Total Area in Jamaica (km²)',
    "Percentage of Jamaica's Area (%)",
    'Area on Bauxite (km²)',
    'Percentage of land use category on Bauxite (%)'
]]

# Display the summary table
display(land_use_bauxite_comparison)

# Optional: Save the result to a CSV file
# land_use_bauxite_comparison.to_csv("land_use_bauxite_comparison_by_category.csv", index=False)

In [ ]:
# Set colours

custom_colors = {
    'Bare Rock': '#A9A9A9',  # Dark Gray (rocky terrain)
    'Agriculture': '#8B4513',  # Dark Brown
    'Freshwater wetland': '#4169E1', #Royal blue
    'Mangrove': '#008080', # Teal Blue
    'Open dry forest': '#A4C639', # Light green 90EE90   
    'Plantation': '#F5DEB3', #  yellow
    'Bauxite extraction / quarry': '#B22222', #Iron Oxide Red
    'Water body': '#4682B4',  # Steel Blue
    'Buildings and other infrastructure': '#000000',  # Black
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',  # lime Green
    'Mixed land use: agriculture and bamboo': '#D2B48C',  #light brown
    'Swamp forest': '#6B8E23',  # Olive Drab
    'Bamboo': '#F4A460',  # Sandy Brown
    'Forest': '#006400',  # Dark Green
}


# Get the list of all categories
all_categories = terrestrial_landcover_area_summary['Land Use Category'].tolist()

# Categories without custom colors
remaining_categories = [cat for cat in all_categories if cat not in custom_colors]

# Create a colormap for remaining categories
#cmap = plt.cm.get_cmap('Set3', len(remaining_categories))
cmap = plt.colormaps['Set3'](len(remaining_categories))

# Assign colors to remaining categories
colormap_colors = {}
for idx, category in enumerate(remaining_categories):
    color = mcolors.rgb2hex(cmap(idx))
    colormap_colors[category] = color

# Combine custom colors with colormap colors
category_colors = {**custom_colors, **colormap_colors}

# Map colors to the GeoDataFrame
terrestrial_landcover['color'] = terrestrial_landcover['Classify'].map(category_colors)


In [ ]:
# Filter for categories with area > 0 km² on bauxite reserves
filtered_bauxite_comparison = land_use_bauxite_comparison[land_use_bauxite_comparison['Area on Bauxite (km²)'] > 0]

# Map colors to the GeoDataFrame
landcover_bauxite['color'] = landcover_bauxite['Category'].map(category_colors)

# Plot the results
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot land cover on bauxite reserves
landcover_bauxite.plot(
    ax=ax,
    color=landcover_bauxite['color'],
    edgecolor='black',
    linewidth=0.1
)

# Plot Jamaica boundary for reference
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--'
)

# Prepare legend handles with detailed information for categories with area > 0
legend_handles = []
for _, row in filtered_bauxite_comparison.iterrows():
    category = row['Category']
    color = category_colors.get(category, '#FFFFFF')  # Default to white if category not found
    area_km2 = row['Area on Bauxite (km²)']
    percentage_on_bauxite = row['Percentage of land use category on Bauxite (%)']
    label = f"{category} ({area_km2:.2f} km², {percentage_on_bauxite:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add legend below the plot
legend = ax.legend(
    handles=legend_handles,
    title="Landcover on Bauxite Reserves (area on bauxite, percentage of land use category on bauxite)",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Add north arrow and scale bar
def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    """
    Add a scale bar to a plot, with "20" under the last tick mark and "km" positioned slightly to the right of the scale bar.
    """
    x, y = location
    bar_half_length = 0.05

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
            ha='center', va='center', fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
            ha='center', va='center', fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
            ha='center', va='center', fontsize=10)

    # Add "km" label slightly to the right of the scale bar
    ax.text(x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
            ha='left', va='center', fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.8), size=0.05, fontsize=12, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=10, headlength=15, width=5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(x, y + size + label_offset, "N", transform=ax.transAxes,
            fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Landcover on Bauxite Reserves",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

#### Protected area bauxite analysis 

In [ ]:
# Proceed with the merge
land_use_bauxite_comparison_with_protected = land_use_bauxite_comparison.merge(
    landcover_bauxite_protected_summary,
    on='Category',
    how='left'
)

In [ ]:
# Rename the columns added from the merge for clarity
land_use_bauxite_comparison_with_protected.rename(
    columns={
        'area_m2': 'area_m2 in Protected Area',
        'area_km2': 'area_km2 in Protected Area'
    },
    inplace=True
)

# Fill NaN values for categories not found in protected areas
land_use_bauxite_comparison_with_protected['area_m2 in Protected Area'] = land_use_bauxite_comparison_with_protected[
    'area_m2 in Protected Area'
].fillna(0)
land_use_bauxite_comparison_with_protected['area_km2 in Protected Area'] = land_use_bauxite_comparison_with_protected[
    'area_km2 in Protected Area'
].fillna(0)

# Calculate the percentage of each land use type on bauxite reserves that is in a protected area
land_use_bauxite_comparison_with_protected['Percentage in Protected Area (%)'] = (
    land_use_bauxite_comparison_with_protected['area_m2 in Protected Area'] /
    land_use_bauxite_comparison_with_protected['Area on Bauxite (m²)'] * 100
)

# Reorder columns for clarity
land_use_bauxite_comparison_with_protected = land_use_bauxite_comparison_with_protected[[
    'Category',
    'Total Area in Jamaica (km²)',
    "Percentage of Jamaica's Area (%)",
    'Area on Bauxite (km²)',
    'Percentage of land use category on Bauxite (%)',
    'area_km2 in Protected Area',
    'Percentage in Protected Area (%)'
]]

# Rename columns for better readability
land_use_bauxite_comparison_with_protected.rename(
    columns={
        'area_km2 in Protected Area': 'Area on Bauxite in Protected Area (km²)',
        'Percentage in Protected Area (%)': 'Percentage of Bauxite Area in Protected Area (%)'
    },
    inplace=True
)

# Display the final table
display(land_use_bauxite_comparison_with_protected)

In [ ]:
# Verify the merged DataFrame
print("Land use categories in protected areas and on bauxite reserves:")
display(land_use_bauxite_comparison_with_protected)

# Set colors for categories
custom_colors = {
    'Bare Rock': '#A9A9A9',
    'Agriculture': '#8B4513',
    'Freshwater wetland': '#4169E1',
    'Mangrove': '#008080',
    'Open dry forest': '#A4C639',
    'Plantation': '#F5DEB3',
    'Bauxite extraction / quarry': '#B22222',
    'Water body': '#4682B4',
    'Buildings and other infrastructure': '#000000',
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',
    'Mixed land use: agriculture and bamboo': '#D2B48C',
    'Swamp forest': '#6B8E23',
    'Bamboo': '#F4A460',
    'Forest': '#006400',
}

# Assign colors to landcover_bauxite_protected
landcover_bauxite_protected['color'] = landcover_bauxite_protected['Category'].map(custom_colors)

# Set gray for the rest of the bauxite reserves not in protected areas
landcover_bauxite_non_protected = landcover_bauxite.overlay(
    landcover_bauxite_protected, how='difference'
)
landcover_bauxite_non_protected['color'] = '#D3D3D3'  # Light gray for non-protected areas

# Create the map
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot protected land use categories
landcover_bauxite_protected.plot(
    ax=ax,
    color=landcover_bauxite_protected['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Area of land use on bauxite that is protected"
)

# Plot non-protected bauxite reserves
landcover_bauxite_non_protected.plot(
    ax=ax,
    color=landcover_bauxite_non_protected['color'],
    edgecolor='black',
    linewidth=0.1,
    label="Non-Protected Bauxite Reserves"
)

# Plot Jamaica boundary
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--'
)

# Prepare legend handles for the categories
legend_handles = []
for _, row in land_use_bauxite_comparison_with_protected.iterrows():
    category = row['Category']
    color = custom_colors.get(category, '#FFFFFF')  # Default white if color not found
    area = row['Area on Bauxite in Protected Area (km²)']
    percentage = row['Percentage of Bauxite Area in Protected Area (%)']
    if area > 0:  # Include only categories with non-zero areas
        label = f"{category} ({area:.2f} km², {percentage:.2f}%)"
        patch = mpatches.Patch(color=color, label=label)
        legend_handles.append(patch)

# Add a legend
legend = ax.legend(
    handles=legend_handles,
    title="Land use types on bauxite that are protected",
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=2,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    prop={'family': 'Times New Roman'}
)

# Add scale bar and north arrow
def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=2, tick_height=0.01, label_offset=0.02, km_offset=0.03):
    x, y = location
    bar_half_length = 0.05

    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],
        transform=ax.transAxes, color='black', linewidth=linewidth
    )
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    ax.text(x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, ha='center', fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, ha='center', fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, ha='center', fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, ha='left', fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.8), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', headwidth=10, headlength=15, width=5)
    )
    ax.text(x, y + size + label_offset, "N", transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center")

add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add title
plt.title(
    "Area of land use on bauxite that is protected",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    loc='center',
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
all_protected_areas = geopandas.read_file(os.path.join(base_path, 'allprotectedareas.gpkg'))
all_protected_areas = all_protected_areas.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
all_protected_areas['area_hectares'] = 0.0001*all_protected_areas.geometry.area # Convert area to hectares
all_protected_areas_total_area = all_protected_areas['area_hectares'].sum()


In [ ]:
landcover_bauxite = landcover \
    .overlay(bauxite.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite
landcover_bauxite.plot()

In [ ]:
landcover_bauxite

In [ ]:
landcover_bauxite_total_area = landcover_bauxite['area_hectares'].sum() # area
print(landcover_bauxite_total_area)

In [ ]:
landcover_bauxite_area_classified = landcover_bauxite[['area_hectares', 'Classify']].groupby('Classify').sum()
landcover_bauxite_area_classified["area_percentage"] = 100.0*landcover_bauxite_area_classified["area_hectares"]/landcover_bauxite_total_area
print(landcover_bauxite_area_classified)
landcover_bauxite_area_classified.to_csv(os.path.join(output_path, 'landcover_bauxite_area_classified.csv'))


In [ ]:
landcover_bauxite_allprotected = landcover_bauxite \
    .overlay(all_protected_areas.set_geometry("geometry"), how='intersection') #intersecting landcover, bauxite with protected areas
landcover_bauxite_allprotected.plot()

In [ ]:
landcover_bauxite_allprotected_total_area = landcover_bauxite_allprotected['area_hectares'].sum() # area
landcover_bauxite_allprotected_total_area